# Q-Learning（Off-Policy）

> Q-Learning Off-Policy 版本引入两个关键变化：其一，**行为策略**（$\pi_b$）与**目标策略**（$\pi_T$）彻底解耦——行为策略采用**均匀随机策略**预先收集轨迹，目标策略持续用 $\max_{a'} Q(s_{t+1}, a')$ 做贪心更新；其二，引入**经验回放**（Replay Buffer）机制，将收集到的轨迹存入经验池，反复复用，而非每轮重新采样。
>
> Q 值更新公式（Off-Policy 特性体现在**数据来源**，而非公式本身）：
>
> $$Q(s_t,a_t) \leftarrow Q(s_t,a_t) - \alpha \left[Q(s_t,a_t) - \left(r_{t+1} + \gamma \max_{a'} Q(s_{t+1},a')\right)\right]$$
>
> On-Policy 与 Off-Policy（本版本）的核心对比如下：
>
> ---

| 对比维度 | On-Policy Q-Learning | Off-Policy Q-Learning（本版本） |
|---|---|---|
| 行为策略 $\pi_b$ | $\varepsilon$-Greedy（与目标策略相同） | 均匀随机（$\frac{1}{5}$ 等概率） |
| 目标策略 $\pi_T$ | $\varepsilon$-Greedy | Greedy（$\arg\max Q$） |
| 数据采集方式 | 每轮重新采样 | 预先填充经验池，反复复用 |
| 数据利用率 | 低（每条轨迹仅用一次） | **高**（历史轨迹可被多轮学习） |
| 时序相关性 | 高（相邻样本高度相关） | **低**（随机采样打破时序相关） |

> ---
>
> **经验回放**（Replay Buffer）的核心思想：先用行为策略一次性收集一批轨迹存入经验池，训练时每轮从经验池中随机采样 $(s,a,r,s')$ 四元组更新 Q 值，同一条轨迹可被多次利用，既提高数据利用率，又通过随机采样打破样本间的时序相关性，是 DQN 等深度强化学习方法的核心组件之一。

## 一、导入库与环境初始化

In [1]:
# 导入 NumPy 库，用于数组操作（Q 表、策略矩阵等）
import numpy as np
# 导入 random 模块，用于随机选择初始状态和动作
import random
# 导入 importlib 标准库，用于按文件路径动态加载模块
import importlib.util
# 导入 os 模块，用于拼接当前目录与文件名
import os
# 导入 time 模块（备用，可控制训练展示节奏）
import time
# 导入 Jupyter 显示控制函数，训练时清除输出以实时刷新展示
from IPython.display import clear_output

# 环境文件名 "02.1.ModelFree_Env_GridWorldV2.py" 以数字开头，
# Python 无法直接 import，需通过 importlib 按路径加载
_env_path = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "02.1.ModelFree_Env_GridWorldV2.py")  # 环境文件的绝对路径（str）
_spec = importlib.util.spec_from_file_location(
    "GridWorld_v2", _env_path)   # 构造模块规格对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)  # 根据规格创建模块对象
_spec.loader.exec_module(GridWorld_v2)                  # 执行模块代码，完成加载

In [2]:
# 设置网格世界行数（必须与 desc 字符串行数一致）
rows = 5
# 设置网格世界列数（必须与 desc 每行字符数一致）
columns = 5

# 创建 5×5 GridWorld 环境实例
# forbiddenAreaScore=-10: 进入障碍格（'#'）的即时惩罚
# score=1: 到达目标格（'T'）的即时正奖励
# desc: 地图描述，'.'=普通格，'#'=障碍格，'T'=目标格
gridworld = GridWorld_v2.GridWorld_v2(
    forbiddenAreaScore=-10, score=1,
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."])

# 显示网格地图（'⬜'=普通格，'🚫'=障碍，'✅'=目标）
gridworld.show()

# 初始化状态价值向量（全零，用于展示）
# shape: (25,)——每个元素对应一个状态（0~24）的价值估计
value = np.zeros(rows * columns)

# 初始化 Q 表（动作价值矩阵，全零）
# shape: (25, 5)——行=状态索引[0~24]，列=5种动作的 Q 值
qtable = np.zeros((rows * columns, 5))

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️


## 二、Q-Learning Off-Policy 算法实现

In [3]:
def QLearning_Off_Policy(gridworld: GridWorld_v2.GridWorld_v2,
                         TrajectoryPoolSize=50,
                         gamma=0.99,
                         trajectorySteps=-1,
                         learning_rate=0.001,
                         num_episodes=600) -> np.ndarray:
    '''
    Q-Learning Off-Policy 算法（含经验回放机制）。

    先用均匀随机的行为策略（behavior policy）预先收集一批轨迹
    存入经验池（Replay Buffer），然后反复从经验池中采样学习，
    使目标策略（target policy）不断改进，直到收敛。

    与 On-Policy 版本的关键区别：
    - 行为策略为均匀随机（1/5 等概率选动作）
    - 目标策略为贪心策略（argmax Q）
    - 数据可被多次复用（Off-Policy 的核心优势）

    参数:
        gridworld (GridWorld_v2): GridWorld 环境实例
        TrajectoryPoolSize (int): 经验池大小（预先收集的轨迹条数）
        gamma (float): 折扣因子，取值 (0,1]
        trajectorySteps (int): 每条轨迹最大步数；-1 表示到目标自动停止
        learning_rate (float): TD 学习率，控制 Q 值更新幅度
        num_episodes (int): 训练轮数（每轮遍历一次经验池并更新 Q 值）

    返回值:
        np.ndarray: 最终收敛的动作价值矩阵（Q 表），
                    shape (25, 5)——行=状态，列=动作
    '''
    # 初始化状态价值向量（全零，仅用于展示）
    # shape: (25,)——每个元素对应一个状态的价值
    state_value = np.zeros((rows * columns))

    # 初始化动作价值矩阵 Q（全零）
    # shape: (25, 5)——行=状态[0~24]，列=5种动作的 Q 值
    # 5 表示 5 种可能的动作（上/下/左/右/原地）
    action_value = np.zeros((rows * columns, 5))

    # 初始化目标策略（随机确定性策略，独热编码）
    # shape: (25, 5)——每行只有一个位置为 1
    policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]

    # 初始化行为策略（均匀随机策略）
    # shape: (25, 5)——每行每列均为 0.2（5 种动作等概率）
    behavior_policy = np.ones((rows * columns, 5)) * 0.2

    epsilon = 0.5   # epsilon 初始值（此处主要用于显示，行为策略固定为均匀随机）

    initState = 10  # 固定初始状态为 10（地图中部）

    TrajectoryPool = []    # 经验池：存储预先收集的轨迹列表
    TrajectoryPoolSize = 50  # 经验池大小：预先收集 50 条轨迹

    # ── 阶段一：使用行为策略预先填充经验池 ──────────────────
    for i in range(TrajectoryPoolSize):  # 循环收集 50 条轨迹

        if trajectorySteps == -1:
            # 设置到达目标时自动停止的标志
            stop_when_reach_target = True

        # 使用均匀随机的行为策略生成轨迹
        # 返回值：列表，每个元素为 (状态, 动作, 奖励, 下一状态, 下一动作) 五元组
        Trajectory = gridworld.getTrajectoryScore(
            nowState=initState,               # 固定起始状态（整数，0~24）
            action=random.randint(0, 4),       # 随机初始动作（整数，0~4）
            policy=behavior_policy,            # 使用均匀随机的行为策略
            steps=trajectorySteps,             # 轨迹最大步数（-1 时由 stop 控制）
            stop_when_reach_target=True)       # 到达目标时停止

        TrajectoryPool.append(Trajectory)  # 将本条轨迹加入经验池

    # ── 阶段二：从经验池中多轮学习，迭代改进目标策略 ──────────
    for episode in range(num_episodes):  # 训练主循环
        print("episode", f"{episode}/{num_episodes}")  # 打印当前轮次

        # 初始化状态访问计数器（长度 25，记录本轮各状态访问次数）
        cnt = [0 for i in range(25)]

        # 遍历经验池中的每一条轨迹
        for Trajectory in TrajectoryPool:
            steps = len(Trajectory) - 1  # 当前轨迹的有效步数

            # 从轨迹末尾向前遍历，执行 Q-Learning 的 TD 更新
            for k in range(steps, -1, -1):
                # 解包当前步的转移信息
                # tmpstate:   当前状态 int
                # tmpaction:  当前动作 int
                # tmpscore:   即时奖励 float
                # nextState:  下一状态 int
                # nextAction: 下一动作 int（Off-Policy Q-Learning 中不使用）
                tmpstate, tmpaction, tmpscore, nextState, nextAction = Trajectory[k]
                cnt[tmpstate] += 1  # 记录状态访问次数

                # Q-Learning TD 误差（Off-Policy 核心）
                # 使用 max Q(s', a')：目标策略总是选最优动作
                # action_value[nextState].max()：shape 标量，下一状态最大 Q 值
                TD_error = (action_value[tmpstate][tmpaction]
                            - (tmpscore + gamma * action_value[nextState].max()))
                # 按 TD 误差更新 Q 值
                action_value[tmpstate][tmpaction] -= learning_rate * TD_error

        # 策略改进：选 Q 值最大的动作作为目标策略
        # np.argmax(action_value, axis=1)：shape (25,)，每个状态最优动作索引
        # np.eye(5)[...]：转为独热编码，shape (25, 5)
        policy = np.eye(5)[np.argmax(action_value, axis=1)]

        print(np.array(cnt).reshape(5, 5))  # 打印 5×5 状态访问次数矩阵

        # 计算状态价值（用目标策略加权 Q 值）
        # V(s) = Σ_a π(a|s)*Q(s,a)，shape (25,)
        state_value = np.sum(policy * action_value, axis=1)
        print(state_value)  # 打印状态价值向量

        # 全局平均状态价值（衡量整体策略质量）
        mean_state_value = np.sum(policy * action_value, axis=1).mean()

        gridworld.showPolicy(policy)                             # 显示当前目标策略
        print(np.round(state_value, decimals=4).reshape(5, 5))  # 打印 5×5 状态价值（保留4位小数）
        print("mean_state_value", mean_state_value)             # 打印平均状态价值

    return action_value  # 返回最终学到的动作价值矩阵（Q 表），shape (25, 5)

## 三、运行 Q-Learning Off-Policy 并保存结果

In [4]:
# 调用 Q-Learning Off-Policy 算法训练智能体
# Off-Policy 版本效率极高：只需收集一次经验池，后续反复学习
# 通常仅需 1 轮经验池就能学到高质量策略，远优于 TD 和 SARSA
# 返回值：action_value，shape (25, 5)——最终 Q 表
action_value = QLearning_Off_Policy(gridworld)

episode 0/600
[[150 173 148 124 116]
 [214 124 123 121 103]
 [268 157  98  86  62]
 [145  78  50  39  49]
 [ 87  51  38  41  43]]
[2.92969138e-26 3.82464441e-24 3.75568706e-22 5.93352350e-20
 1.24228382e-17 1.32468108e-28 2.46551088e-26 6.50782061e-20
 8.36308784e-18 3.00864205e-15 9.69577784e-31 3.32155966e-33
 1.79039285e-02 1.85345373e-15 7.03898371e-13 6.20854522e-33
 1.59564249e-02 9.99955001e-03 7.01162520e-03 1.86698230e-10
 1.32939802e-35 5.62532332e-05 9.00258924e-03 4.64118842e-05
 1.03587429e-07]
➡️➡️➡️➡️⬇️
⬆️⏫️⏩️➡️⬇️
⬆️⬅️⏬➡️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
[[0.     0.     0.     0.     0.    ]
 [0.     0.     0.     0.     0.    ]
 [0.     0.     0.0179 0.     0.    ]
 [0.     0.016  0.01   0.007  0.    ]
 [0.     0.0001 0.009  0.     0.    ]]
mean_state_value 0.002399075471393379
episode 1/600
[[150 173 148 124 116]
 [214 124 123 121 103]
 [268 157  98  86  62]
 [145  78  50  39  49]
 [ 87  51  38  41  43]]
[1.11627331e-23 1.23550370e-21 9.80659762e-20 1.13199833e-17
 1.57803259e-15